**Author**:
- Tianci Wang - [tiancwang@ethz.ch](tianci:tiancwang@ethz.ch)

**Date**: 06/08/2026

# Policy Portfolio Construction Code v1.0
This code is used to perform policy evaluation and build policy portfolios for each energy transition pathway based on the baseline transition pathway of EP2050+.
The basic assumption is that the current in-force policies are sufficient for the transition following the baseline scenario in the near-term period (2030–2040). With this model, we will have not only the current in-force policies but also the possible policies that may be needed in the future, and analyze all policies together to gain an understanding of the policy portfolio construction for different pathways and how the priority of each policy would change over the longer-term period (2040–2050).
## Workflow
The code consists of five main steps:
1. Data preparation:

   Import all the input data. Build a "Policy_evaluation" file for the following MCDA analysis based on the characteristics of the policies, the experts' opinions, the technologies each policy is intended to support, and the factual data of these technologies from the present to 2050.
2. MCDA conducting:

   Calculate the MCDA scores based on the "Policy_evaluation" file by choosing the TOPSIS method for both time periods (2030–2040 and 2040–2050).
3. Relevance Score calculation:

   Calculate the relevance score for each policy based on the comparison of the technology deployment levels between the EP2050+ scenario and the MIX, REMIX, and H2 scenarios, as well as the technologies each policy is intended to support.
4. Robustness analysis:

   Include both uncertainty analysis and sensitivity analysis. For the uncertainty analysis, consider uncertainties in the technology data, differences in experts' opinions, and the allocation of the policy support levels to different technologies, and use Monte Carlo simulation to run a sufficient number of iterations and observe the variation in the results. For the sensitivity analysis, only consider the sensitivity of the TOPSIS criteria weights. Analyze how much the weights can change without leading to a change in the ranking for both time periods.
5. Result visualization:

   Generate the final plots to present the relevance scores and TOPSIS scores, and generate the policy portfolio results along with the visualization of the robustness analysis results.
## Input data
1. Policy related data: Policy characteristic (Policy_data.csv), Experts' opinions (Criteria_Acceptance_Ins.csv, Criteria_Admin_Burden.csv, Criteria_Perceived_Equity.csv) ;
2. Technology related data: Technology factual data (Technology_data.csv).
3. Scenario data: The energy system results from the scenario simulation (Scenario_energy_production.csv, Scenario_installed_capacity.csv);
4. Others: Technology policy relationship (Technology_policy_matrix.csv) .

## 1 Data Preparation
* Load all input files, verify structure and technology-name consistency.
* Build criteria data, each tested individually before combining.
  * Cost of carbon abatement: For each policy, take the technologies it supports (from Technology_policy_matrix), normalize their weights to sum to 1, then compute a weighted average of Cost_current (for 2030-2040) and Cost_2050 (for 2040-2050) from Technology_data.
  * Social Acceptance of technologies: Same logic as Cost, just a different source column. So this value is the SAME for both time periods.
  * Deployment difficulty:
    * For the EP2050+ baseline scenario, compute each technology's SHARE of
     its own sector's total production (so shares sum to 1 within a sector).
    * For each policy, take the weighted average of these shares across the
     technologies it supports (using the same normalized policy-technology
     weights as the cost or acceptance of technologies).
    * Same value for both time periods.
    * Interpretation: LOW share = technology is barely present in the baseline
     -> policy needs to work HARDER to promote it -> HIGH deployment difficulty.
     This direction (low value = harder) will be handled later by TOPSIS.
  * Cost gap: Cost_current minus Cost_2050. Only used in the SECOND TOPSIS run (2040-2050).
  * Acceptance of policy instrument, Perceived equity, Administrative burden (Qualitative expert-opinion criteria): Look up the policy's category (Instrument / Administrative_touchpoint / their combo) in a small reference table and pull the 'mode' value for main TOPSIS. Same value for BOTH time periods.
* Assemble two clean Policy_evaluation tables, ready for TOPSIS.
  * Period 1 (2030-2040): 6 criteria, using Cost_current
  * Period 2 (2040-2050): 7 criteria, using Cost_2050, PLUS Cost gap

In [3]:
# Data Import

import pandas as pd
import numpy as np

## Create file paths
DATA_DIR = "D:/Tansy/Master thesis/MCDA/data/final/input/"  # <-- change this to your folder

files = {
    "policy": "Policy_data.csv",
    "tech_policy_matrix": "Technology_policy_matrix.csv",
    "technology": "Technology_data.csv",
    "scenario_production": "Scenario_energy_production.csv",
    "scenario_capacity": "Scenario_installed_capacity.csv",
    "crit_acceptance_ins": "Criteria_Acceptance_Ins.csv",
    "crit_admin_burden": "Criteria_Admin_Burden.csv",
    "crit_perceived_equity": "Criteria_Perceived_Equity.csv",
}

## Load everything into a dictionary of DataFrames
data = {}
for key, filename in files.items():
    data[key] = pd.read_csv(DATA_DIR + filename)

## Sanity check on each table
print("=" * 70)
for key, df in data.items():
    print(f"[{key}]  ({files[key]})")
    print(f"  shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"  columns: {list(df.columns)}")
    n_missing = df.isna().sum().sum()
    print(f"  total missing values: {n_missing}")
    print("-" * 70)

[policy]  (Policy_data.csv)
  shape: 48 rows x 9 columns
  columns: ['Policy_ID', 'Status', 'Order', 'Name', 'Source_policies', 'Energy_sector', 'Promoting_technologies', 'Instrument', 'Administrative_touchpoint']
  total missing values: 0
----------------------------------------------------------------------
[tech_policy_matrix]  (Technology_policy_matrix.csv)
  shape: 48 rows x 20 columns
  columns: ['Policy_ID', 'Biomass_Boiler', 'Heat_Pump', 'Methane_Boiler', 'CHP_waste_heating', 'CHP_Methane_heating', 'CCGT_Ren_Methane', 'CHP_waste_electricity', 'CHP_Methane_electricity', 'Hydro_reservoir', 'Hydro_run_of', 'PV_Roof', 'Pumped_Hydro', 'Wind', 'CCGT_Gas_DACCS', 'Alpine_PV', 'Heavy_EV', 'Heavy_FCEV', 'Light_EV', 'Light_FCEV']
  total missing values: 768
----------------------------------------------------------------------
[technology]  (Technology_data.csv)
  shape: 19 rows x 4 columns
  columns: ['Technology', 'Acceptance', 'Cost_current', 'Cost_2050']
  total missing values: 0
----

In [4]:
# Build Policy_evaluation table

## Criterion 1: Cost of carbon abatement (EUR/tCO2eq)

### Take the Technology_policy_matrix and normalize each row so its non-missing weights sum to 1. Rows that are all-NaN stay all-NaN.
def normalize_weights(matrix_df, id_col="Policy_ID"):
    tech_cols = [c for c in matrix_df.columns if c != id_col]
    weights = matrix_df.set_index(id_col)[tech_cols]
    row_sums = weights.sum(axis=1, skipna=True)
    normalized = weights.div(row_sums, axis=0)
    return normalized  # index = Policy_ID, columns = technologies, values = normalized weights

### For each policy, compute sum( weight_i * cost_i )  over technologies i the policy supports
def compute_weighted_cost(normalized_weights, tech_df, cost_col):
    # Build a lookup: technology name -> cost value
    cost_lookup = tech_df.set_index("Technology")[cost_col]
    # Reindex so column order in normalized_weights matches cost_lookup order， any tech name mismatch becomes a NaN column。
    aligned_costs = cost_lookup.reindex(normalized_weights.columns)
    # Weighted sum
    weighted_cost = normalized_weights.mul(aligned_costs, axis=1).sum(axis=1, skipna=True)
    return weighted_cost

### Run it
weights = normalize_weights(data["tech_policy_matrix"])

### Sanity check
tech_names_in_data = set(data["technology"]["Technology"])
tech_names_in_matrix = set(weights.columns)
missing_names = tech_names_in_matrix - tech_names_in_data
if missing_names:
    print(f"WARNING: these matrix columns have NO match in Technology_data.csv: {missing_names}")
else:
    print("All technology names in the matrix match Technology_data.csv")

cost_2030_2040 = compute_weighted_cost(weights, data["technology"], "Cost_current")
cost_2040_2050 = compute_weighted_cost(weights, data["technology"], "Cost_2050")

### Print results
print("\nFirst 5 policies - Cost of carbon abatement (EUR/tCO2eq):")
preview = pd.DataFrame({
    "Cost_2030_2040": cost_2030_2040,
    "Cost_2040_2050": cost_2040_2050,
}).head()
print(preview)

### Check NaN results
n_nan = cost_2030_2040.isna().sum()
print(f"\nPolicies with missing Cost result: {n_nan} out of {len(cost_2030_2040)}")

## Criterion 2: Social Acceptance of technologies

### Run it again
acceptance_score = compute_weighted_cost(weights, data["technology"], "Acceptance")
acceptance_2030_2040 = acceptance_score
acceptance_2040_2050 = acceptance_score

print("\nFirst 5 policies - Social Acceptance of technologies (same both periods):")
preview2 = pd.DataFrame({
    "Acceptance_2030_2040": acceptance_2030_2040,
    "Acceptance_2040_2050": acceptance_2040_2050,
}).head()
print(preview2)

n_nan2 = acceptance_score.isna().sum()
print(f"\nPolicies with missing Acceptance result: {n_nan2} out of {len(acceptance_score)}")

All technology names in the matrix match Technology_data.csv

First 5 policies - Cost of carbon abatement (EUR/tCO2eq):
           Cost_2030_2040  Cost_2040_2050
Policy_ID                                
I_01          4749.681785     2950.054383
N_02          2560.649435     1347.986975
I_03          2573.741835     1648.083733
I_04          4749.681785     2950.054383
I_05          2560.649435     1347.986975

Policies with missing Cost result: 0 out of 48

First 5 policies - Social Acceptance of technologies (same both periods):
           Acceptance_2030_2040  Acceptance_2040_2050
Policy_ID                                            
I_01                      0.714                 0.714
N_02                      0.235                 0.235
I_03                      0.660                 0.660
I_04                      0.714                 0.714
I_05                      0.235                 0.235

Policies with missing Acceptance result: 0 out of 48


In [5]:
## Criterion 3: Deployment level under baseline pathway (aka deployment difficulty)

### Compute each technology's share of its sector's EP2050+ total
sector_totals = data["scenario_production"].groupby("Sector")["EP2050+"].transform("sum")
data["scenario_production"]["EP2050_share"] = data["scenario_production"]["EP2050+"] / sector_totals

print("\nSanity check - shares should sum to 1.0 within each sector:")
print(data["scenario_production"].groupby("Sector")["EP2050_share"].sum())

### Weighted average of these shares across each policy's supported technologies (reuse the compute_weighted_cost function)
deployment_level = compute_weighted_cost(weights, data["scenario_production"], "EP2050_share")
deployment_level_2030_2040 = deployment_level
deployment_level_2040_2050 = deployment_level

print("\nFirst 5 policies - Deployment level (baseline share):")
print(deployment_level.head())

n_nan3 = deployment_level.isna().sum()
print(f"\nPolicies with missing Deployment level result: {n_nan3} out of {len(deployment_level)}")

### Criterion 7: Cost gap
cost_gap = cost_2030_2040 - cost_2040_2050

print("\nFirst 5 policies - Cost gap (EUR/tCO2eq saved by 2050):")
print(cost_gap.head())


Sanity check - shares should sum to 1.0 within each sector:
Sector
Electricity    1.0
Heating        1.0
Transport      1.0
Name: EP2050_share, dtype: float64

First 5 policies - Deployment level (baseline share):
Policy_ID
I_01    0.196376
N_02    0.250000
I_03    0.033813
I_04    0.196376
I_05    0.250000
dtype: float64

Policies with missing Deployment level result: 0 out of 48

First 5 policies - Cost gap (EUR/tCO2eq saved by 2050):
Policy_ID
I_01    1799.627402
N_02    1212.662460
I_03     925.658101
I_04    1799.627402
I_05    1212.662460
dtype: float64


In [7]:
## Criteria 4-6: Acceptance of policy instrument, Perceived equity, Administrative burden (Qualitative expert-opinion criteria)

### Build the lookup function
def lookup_by_category(policy_df, category_col, ref_df, ref_category_col, value_col="mode"):
    # Standardize categories' names on BOTH sides before matching
    policy_key = policy_df[category_col].str.strip().str.lower()
    ref_key = ref_df[ref_category_col].str.strip().str.lower()

    # Build a lookup: normalized category name -> mode value
    value_lookup = pd.Series(ref_df[value_col].values, index=ref_key)

    result = policy_key.map(value_lookup)
    result.index = policy_df["Policy_ID"]
    return result

policy_df = data["policy"]

### Criterion 4: Acceptance of policy instrument
acceptance_ins = lookup_by_category(
    policy_df, "Instrument",
    data["crit_acceptance_ins"], "Instrument",
)

# Criterion 5: Administrative burden
admin_burden = lookup_by_category(
    policy_df, "Administrative_touchpoint",
    data["crit_admin_burden"], "Administrative_touchpoint",
)

# Criterion 6: Perceived equity
policy_df = policy_df.copy()
policy_df["Instrument_Administrative_touchpoint"] = (
    policy_df["Instrument"] + "_" + policy_df["Administrative_touchpoint"]
)
perceived_equity = lookup_by_category(
    policy_df, "Instrument_Administrative_touchpoint",
    data["crit_perceived_equity"], "Instrument_Administrative_touchpoint",
)

### Check for any unmatched (NaN) results
for name, series in [
    ("Acceptance of policy instrument", acceptance_ins),
    ("Administrative burden", admin_burden),
    ("Perceived equity", perceived_equity),
]:
    n_missing = series.isna().sum()
    status = "all matched" if n_missing == 0 else f"{n_missing} unmatched!"
    print(f"{name}: {status}")

print("\nFirst 5 policies - all 3 qualitative criteria:")
print(pd.DataFrame({
    "Acceptance_Ins": acceptance_ins,
    "Admin_Burden": admin_burden,
    "Perceived_Equity": perceived_equity,
}).head())

Acceptance of policy instrument: all matched
Administrative burden: all matched
Perceived equity: all matched

First 5 policies - all 3 qualitative criteria:
           Acceptance_Ins  Admin_Burden  Perceived_Equity
Policy_ID                                                
I_01                    1             3                 4
N_02                    1             3                 4
I_03                    4             3                 2
I_04                    1             3                 4
I_05                    1             3                 4


In [10]:
# Assemble Policy_evaluation tables

policy_evaluation_2030_2040 = pd.DataFrame({
    "Cost_carbon_abatement": cost_2030_2040,
    "Social_acceptance_tech": acceptance_2030_2040,
    "Deployment_difficulty": deployment_level_2030_2040,
    "Acceptance_policy_instrument": acceptance_ins,
    "Perceived_equity": perceived_equity,
    "Administrative_burden": admin_burden,
})

policy_evaluation_2040_2050 = pd.DataFrame({
    "Cost_carbon_abatement": cost_2040_2050,
    "Social_acceptance_tech": acceptance_2040_2050,
    "Deployment_difficulty": deployment_level_2040_2050,
    "Acceptance_policy_instrument": acceptance_ins,
    "Perceived_equity": perceived_equity,
    "Administrative_burden": admin_burden,
    "Cost_gap": cost_gap,  # extra column, not used in TOPSIS yet
})

print("\nPolicy_evaluation_2030_2040 (first 5 rows):")
print(policy_evaluation_2030_2040.head())
print(f"\nShape: {policy_evaluation_2030_2040.shape}")
print(f"Any missing values? {policy_evaluation_2030_2040.isna().sum().sum()}")

print("\n" + "-" * 70)
print("\nPolicy_evaluation_2040_2050 (first 5 rows):")
print(policy_evaluation_2040_2050.head())
print(f"\nShape: {policy_evaluation_2040_2050.shape}")
print(f"Any missing values? {policy_evaluation_2040_2050.isna().sum().sum()}")

# Save to CSV
policy_evaluation_2030_2040.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/intermediate/Policy_evaluation_2030_2040.csv",
    index=True)  # <-- change this to your folder
policy_evaluation_2040_2050.to_csv(r"D:/Tansy/Master thesis/MCDA/data/final/intermediate/policy_evaluation_2040_2050.csv",
    index=True)  # <-- change this to your folder
print("\n✅ Saved: Policy_evaluation_2030_2040.csv, Policy_evaluation_2040_2050.csv")


Policy_evaluation_2030_2040 (first 5 rows):
           Cost_carbon_abatement  Social_acceptance_tech  \
Policy_ID                                                  
I_01                 4749.681785                   0.714   
N_02                 2560.649435                   0.235   
I_03                 2573.741835                   0.660   
I_04                 4749.681785                   0.714   
I_05                 2560.649435                   0.235   

           Deployment_difficulty  Acceptance_policy_instrument  \
Policy_ID                                                        
I_01                    0.196376                             1   
N_02                    0.250000                             1   
I_03                    0.033813                             4   
I_04                    0.196376                             1   
I_05                    0.250000                             1   

           Perceived_equity  Administrative_burden  
Policy_ID         